# 04 · Decision-threshold control 
Question: can the held-out-institution gain of the connectivity losses be obtained simply by shifting the stem decision threshold of the baseline?

Specification (fixed before running; see the decision log):
- All 72 native-resolution runs on the random and region splits (3 losses × 4 architectures × 3 seeds × 2 splits).
- A scalar offset *b* is added to the stem logit before argmax, *b* ∈ {−2.0, −1.5, …, +8.0}.
- *b* is chosen per run on that split's **validation** set (random: 110 images; region: UTokyo) by maximising stem IoU; ties → smallest |*b*|.
- The **test** set is evaluated once at the chosen *b* → `metrics_thr.csv`, `summary_thr.json`, `offset_val.csv` in each run folder.

Resumable: finished runs are skipped. Runtime: about 1 minute per run on an A100.

In [ ]:
# --- Setup (run every session) ---
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/gwfss_stem_seg'
DATA_ROOT = '/content/gwfss'                      # local disk: fast I/O, rebuilt each session
SPLIT_FILE = f'{PROJECT}/splits/splits.json'      # frozen once, shared by all runs
RESULTS = f'{PROJECT}/results'                    # checkpoints, logs, metrics (on Drive)

import sys, os
sys.path.insert(0, f'{PROJECT}/src')
!pip -q install segmentation-models-pytorch==0.5.0
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from gwfss_stem import data
manifest = data.prepare_dataset(DATA_ROOT)   # downloads GWFSS_v1.0 from Hugging Face (~600 MB) if not present
print(len(manifest), 'images')

In [ ]:
import glob, json
from gwfss_stem import engine
print('offset grid:', engine.OFFSETS)
runs = []
for d in sorted(os.path.dirname(p) for p in glob.glob(f'{RESULTS}/*/*/*/*/*/final.pt')):
    cfg = json.load(open(f'{d}/config.json'))['config']
    if cfg['split'] in ('random', 'region') and cfg['scale'] == 1.0:
        runs.append(d)
print(len(runs), 'runs (expected 72)')
for d in runs:
    engine.evaluate_offset(d, DATA_ROOT, SPLIT_FILE)